## IEOR4004 Project II — Q1 & Q2 (Scheduling the NBA)

This notebook:
- computes Q1 (a)–(d) from `games.csv`
- builds/solves the Q2 feasibility integer program (no meaningful objective)
- exports a feasible schedule to CSV (`Date, Home, Visitor`)

In [12]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import pandas as pd

# -------------------------
# Data loading + Q1 helpers
# -------------------------


@dataclass
class Q1Data:
    teams: list[str]
    dates: list[str]
    home_on: dict[tuple[str, str], int]
    away_on: dict[tuple[str, str], int]
    home_vs: dict[tuple[str, str], int]
    away_vs: dict[tuple[str, str], int]
    home_dates: dict[str, list[str]]
    away_dates: dict[str, list[str]]


def load_games(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    expected_cols = {"Date", "Visitor", "Home", "Attend.", "Arena", "Notes"}
    missing = expected_cols.difference(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns in games.csv: {sorted(missing)}")

    df["Date_dt"] = pd.to_datetime(df["Date"], format="%a, %b %d, %Y")
    df["Date_label"] = df["Date_dt"].dt.strftime("%a, %b %d, %Y")

    df["Attendance"] = (
        df["Attend."]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace({"": None, "nan": None})
        .astype("float")
    )

    return df


def compute_q1_data(df: pd.DataFrame) -> Q1Data:
    teams = sorted(set(df["Home"]).union(set(df["Visitor"])))
    dates = sorted(
        df["Date_label"].unique(),
        key=lambda x: pd.to_datetime(x, format="%a, %b %d, %Y"),
    )

    home_dates = {
        t: sorted(
            df.loc[df["Home"] == t, "Date_label"].unique(),
            key=lambda x: pd.to_datetime(x, format="%a, %b %d, %Y"),
        )
        for t in teams
    }
    away_dates = {
        t: sorted(
            df.loc[df["Visitor"] == t, "Date_label"].unique(),
            key=lambda x: pd.to_datetime(x, format="%a, %b %d, %Y"),
        )
        for t in teams
    }

    home_on = {(i, d): int(d in set(home_dates[i])) for i in teams for d in dates}
    away_on = {(i, d): int(d in set(away_dates[i])) for i in teams for d in dates}

    home_counts_df = (
        pd.crosstab(df["Home"], df["Visitor"])\
        .reindex(index=teams, columns=teams, fill_value=0)
    )
    away_counts_df = (
        pd.crosstab(df["Visitor"], df["Home"])\
        .reindex(index=teams, columns=teams, fill_value=0)
    )

    home_vs = {(i, j): int(home_counts_df.loc[i, j]) for i in teams for j in teams if i != j}
    away_vs = {(i, j): int(away_counts_df.loc[i, j]) for i in teams for j in teams if i != j}

    return Q1Data(
        teams=teams,
        dates=dates,
        home_on=home_on,
        away_on=away_on,
        home_vs=home_vs,
        away_vs=away_vs,
        home_dates=home_dates,
        away_dates=away_dates,
    )


def export_q1_outputs(q1: Q1Data, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    teams = q1.teams

    home_matrix = pd.DataFrame(0, index=teams, columns=teams, dtype=int)
    away_matrix = pd.DataFrame(0, index=teams, columns=teams, dtype=int)
    for i in teams:
        for j in teams:
            if i == j:
                continue
            home_matrix.loc[i, j] = q1.home_vs[(i, j)]
            away_matrix.loc[i, j] = q1.away_vs[(i, j)]

    home_matrix.to_csv(out_dir / "q1_home_vs_counts_matrix.csv", index=True)
    away_matrix.to_csv(out_dir / "q1_away_vs_counts_matrix.csv", index=True)

    long_rows = []
    for i in teams:
        for j in teams:
            if i == j:
                continue
            long_rows.append(
                {
                    "team_i": i,
                    "team_j": j,
                    "home_vs_j_count": q1.home_vs[(i, j)],
                    "away_at_j_count": q1.away_vs[(i, j)],
                }
            )
    pd.DataFrame(long_rows).to_csv(out_dir / "q1_pair_counts_long.csv", index=False)

    home_dates_rows = [{"team": i, "date": d} for i in teams for d in q1.home_dates[i]]
    away_dates_rows = [{"team": i, "date": d} for i in teams for d in q1.away_dates[i]]
    pd.DataFrame(home_dates_rows).to_csv(out_dir / "q1_home_dates.csv", index=False)
    pd.DataFrame(away_dates_rows).to_csv(out_dir / "q1_away_dates.csv", index=False)


# ---------
# File paths
# ---------
# Teammates may run this notebook from different working directories.
# We auto-locate the repo-relative file: Opt Models/Assignment/Project_2/games.csv

_repo_rel_games = Path("Opt Models/Assignment/Project_2/games.csv")

GAMES_CSV = None
for base in [Path.cwd(), *Path.cwd().parents]:
    cand = base / _repo_rel_games
    if cand.exists():
        GAMES_CSV = cand
        break

if GAMES_CSV is None:
    # Fall back: maybe they are already in the Project_2 directory
    cand = Path.cwd() / "games.csv"
    if cand.exists():
        GAMES_CSV = cand

if GAMES_CSV is None:
    raise FileNotFoundError(
        "Could not find games.csv. Tried searching for: "
        f"{_repo_rel_games} up the directory tree, and also ./games.csv. "
        "Please set GAMES_CSV manually."
    )

PROJECT2_DIR = GAMES_CSV.parent
OUT_DIR = PROJECT2_DIR / "outputs_q1_q2"


df = load_games(GAMES_CSV)
q1 = compute_q1_data(df)

len(q1.teams), len(q1.dates)

(16, 16)

In [13]:
# What does `q1` look like?
# `q1` is a Q1Data dataclass (not a single DataFrame). Here are DataFrame views.

q1  # shows fields in the dataclass

Q1Data(teams=['Atlanta Hawks', 'Boston Celtics', 'Brooklyn Nets', 'Chicago Bulls', 'Cleveland Cavaliers', 'Dallas Mavericks', 'Denver Nuggets', 'Golden State Warriors', 'Houston Rockets', 'Los Angeles Lakers', 'Miami Heat', 'Milwaukee Bucks', 'New York Knicks', 'Philadelphia 76ers', 'Phoenix Suns', 'Toronto Raptors'], dates=['Sat, Nov 01, 2025', 'Mon, Nov 03, 2025', 'Wed, Nov 05, 2025', 'Fri, Nov 07, 2025', 'Tue, Nov 11, 2025', 'Thu, Nov 13, 2025', 'Sat, Nov 15, 2025', 'Mon, Nov 17, 2025', 'Wed, Nov 19, 2025', 'Fri, Nov 21, 2025', 'Sun, Nov 23, 2025', 'Thu, Nov 27, 2025', 'Fri, Nov 28, 2025', 'Sat, Nov 29, 2025', 'Mon, Dec 01, 2025', 'Thu, Dec 25, 2025'], home_on={('Atlanta Hawks', 'Sat, Nov 01, 2025'): 0, ('Atlanta Hawks', 'Mon, Nov 03, 2025'): 1, ('Atlanta Hawks', 'Wed, Nov 05, 2025'): 0, ('Atlanta Hawks', 'Fri, Nov 07, 2025'): 1, ('Atlanta Hawks', 'Tue, Nov 11, 2025'): 0, ('Atlanta Hawks', 'Thu, Nov 13, 2025'): 0, ('Atlanta Hawks', 'Sat, Nov 15, 2025'): 1, ('Atlanta Hawks', 'Mon, No

In [14]:
# Long-format (i,j) table like you described

pair_long_df = pd.DataFrame(
    [
        {
            "team_i": i,
            "team_j": j,
            "home_vs_j_count": q1.home_vs[(i, j)],
            "away_at_j_count": q1.away_vs[(i, j)],
        }
        for i in q1.teams
        for j in q1.teams
        if i != j
    ]
)

pair_long_df.head(20)

,team_i,team_j,home_vs_j_count,away_at_j_count
0,Atlanta Hawks,Boston Celtics,1,0
1,Atlanta Hawks,Brooklyn Nets,0,1
2,Atlanta Hawks,Chicago Bulls,1,1
3,Atlanta Hawks,Cleveland Cavaliers,0,1
4,Atlanta Hawks,Dallas Mavericks,1,0
5,Atlanta Hawks,Denver Nuggets,0,1
6,Atlanta Hawks,Golden State Warriors,1,0
7,Atlanta Hawks,Houston Rockets,0,1
8,Atlanta Hawks,Los Angeles Lakers,1,0
9,Atlanta Hawks,Miami Heat,1,0


In [15]:
# Wide matrices (same info as pair_long_df but in matrix form)

home_vs_matrix = pd.DataFrame(0, index=q1.teams, columns=q1.teams, dtype=int)
away_at_matrix = pd.DataFrame(0, index=q1.teams, columns=q1.teams, dtype=int)

for i in q1.teams:
    for j in q1.teams:
        if i == j:
            continue
        home_vs_matrix.loc[i, j] = q1.home_vs[(i, j)]
        away_at_matrix.loc[i, j] = q1.away_vs[(i, j)]

home_vs_matrix, away_at_matrix

(                       Atlanta Hawks  Boston Celtics  Brooklyn Nets  \
 Atlanta Hawks                      0               1              0   
 Boston Celtics                     0               0              0   
 Brooklyn Nets                      1               1              0   
 Chicago Bulls                      1               1              1   
 Cleveland Cavaliers                1               1              1   
 Dallas Mavericks                   0               0              0   
 Denver Nuggets                     1               1              1   
 Golden State Warriors              0               1              0   
 Houston Rockets                    1               1              1   
 Los Angeles Lakers                 0               1              1   
 Miami Heat                         0               0              0   
 Milwaukee Bucks                    0               0              1   
 New York Knicks                    0               1           

In [16]:
# Home/away date lists as DataFrames

home_dates_df = pd.DataFrame(
    [{"team": t, "date": d} for t in q1.teams for d in q1.home_dates[t]]
)
home_dates_df["date_dt"] = pd.to_datetime(home_dates_df["date"], format="%a, %b %d, %Y")
home_dates_df = home_dates_df.sort_values(["team", "date_dt"]).drop(columns=["date_dt"]).reset_index(drop=True)

away_dates_df = pd.DataFrame(
    [{"team": t, "date": d} for t in q1.teams for d in q1.away_dates[t]]
)
away_dates_df["date_dt"] = pd.to_datetime(away_dates_df["date"], format="%a, %b %d, %Y")
away_dates_df = away_dates_df.sort_values(["team", "date_dt"]).drop(columns=["date_dt"]).reset_index(drop=True)

home_dates_df.head(10), away_dates_df.head(10)

(            team               date
 0  Atlanta Hawks  Mon, Nov 03, 2025
 1  Atlanta Hawks  Fri, Nov 07, 2025
 2  Atlanta Hawks  Sat, Nov 15, 2025
 3  Atlanta Hawks  Mon, Nov 17, 2025
 4  Atlanta Hawks  Wed, Nov 19, 2025
 5  Atlanta Hawks  Sun, Nov 23, 2025
 6  Atlanta Hawks  Thu, Nov 27, 2025
 7  Atlanta Hawks  Fri, Nov 28, 2025
 8  Atlanta Hawks  Sat, Nov 29, 2025
 9  Atlanta Hawks  Thu, Dec 25, 2025,
              team               date
 0   Atlanta Hawks  Sat, Nov 01, 2025
 1   Atlanta Hawks  Wed, Nov 05, 2025
 2   Atlanta Hawks  Tue, Nov 11, 2025
 3   Atlanta Hawks  Thu, Nov 13, 2025
 4   Atlanta Hawks  Fri, Nov 21, 2025
 5   Atlanta Hawks  Mon, Dec 01, 2025
 6  Boston Celtics  Mon, Nov 03, 2025
 7  Boston Celtics  Wed, Nov 05, 2025
 8  Boston Celtics  Tue, Nov 11, 2025
 9  Boston Celtics  Sat, Nov 15, 2025)

In [17]:
# Q1 (a) and (d): chronological display
# Reuse the chronologically sorted DataFrames created above.

home_dates_df.head(10), away_dates_df.head(10)

(            team               date
 0  Atlanta Hawks  Mon, Nov 03, 2025
 1  Atlanta Hawks  Fri, Nov 07, 2025
 2  Atlanta Hawks  Sat, Nov 15, 2025
 3  Atlanta Hawks  Mon, Nov 17, 2025
 4  Atlanta Hawks  Wed, Nov 19, 2025
 5  Atlanta Hawks  Sun, Nov 23, 2025
 6  Atlanta Hawks  Thu, Nov 27, 2025
 7  Atlanta Hawks  Fri, Nov 28, 2025
 8  Atlanta Hawks  Sat, Nov 29, 2025
 9  Atlanta Hawks  Thu, Dec 25, 2025,
              team               date
 0   Atlanta Hawks  Sat, Nov 01, 2025
 1   Atlanta Hawks  Wed, Nov 05, 2025
 2   Atlanta Hawks  Tue, Nov 11, 2025
 3   Atlanta Hawks  Thu, Nov 13, 2025
 4   Atlanta Hawks  Fri, Nov 21, 2025
 5   Atlanta Hawks  Mon, Dec 01, 2025
 6  Boston Celtics  Mon, Nov 03, 2025
 7  Boston Celtics  Wed, Nov 05, 2025
 8  Boston Celtics  Tue, Nov 11, 2025
 9  Boston Celtics  Sat, Nov 15, 2025)

In [18]:
# Q1 (b) and (c): home-vs and away-at opponent count matrices (wide tables)

teams = q1.teams

home_vs_matrix = pd.DataFrame(0, index=teams, columns=teams, dtype=int)
away_at_matrix = pd.DataFrame(0, index=teams, columns=teams, dtype=int)

for i in teams:
    for j in teams:
        if i == j:
            continue
        home_vs_matrix.loc[i, j] = q1.home_vs[(i, j)]
        away_at_matrix.loc[i, j] = q1.away_vs[(i, j)]

home_vs_matrix, away_at_matrix

(                       Atlanta Hawks  Boston Celtics  Brooklyn Nets  \
 Atlanta Hawks                      0               1              0   
 Boston Celtics                     0               0              0   
 Brooklyn Nets                      1               1              0   
 Chicago Bulls                      1               1              1   
 Cleveland Cavaliers                1               1              1   
 Dallas Mavericks                   0               0              0   
 Denver Nuggets                     1               1              1   
 Golden State Warriors              0               1              0   
 Houston Rockets                    1               1              1   
 Los Angeles Lakers                 0               1              1   
 Miami Heat                         0               0              0   
 Milwaukee Bucks                    0               0              1   
 New York Knicks                    0               1           

In [19]:
# Optional but recommended: export Q1 tables to CSV (helpful for report/submission)
export_q1_outputs(q1, OUT_DIR)
OUT_DIR

PosixPath('outputs_q1_q2')

## Q2 — Feasibility Integer Program (lecture-aligned)

Decision variable (binary):

- \(x_{i,j,d} = 1\) if on date \(d\), team \(i\) plays **home** vs team \(j\); else 0.

Constraints enforce:
- each team’s **home dates** match Q1(a)
- each team’s **away dates** match Q1(d)
- **home-vs** opponent totals match Q1(b)
- **away-at** opponent totals match Q1(c)

### Note about solver choice
When using Gurobi with the size-limited (restricted) license, the Q2 model triggers:

- `GurobiError: Model too large for size-limited license`

So in this notebook we solve the same IP using an **open-source MILP solver** (PuLP + CBC). The IP formulation (variables/constraints) is unchanged; only the solver backend is different.

In [20]:
# Q2 (open-source solver): PuLP + CBC
# Same lecture-style IP: binary x[i,j,d] and constraints (e)-(h)

import sys

try:
    import pulp
except Exception:
    # If PuLP isn't installed in your kernel environment, install it.
    # (CBC comes bundled with PuLP on most platforms.)
    !{sys.executable} -m pip -q install pulp
    import pulp


def build_q2_pulp_model(q1: Q1Data) -> tuple[pulp.LpProblem, dict[tuple[str, str, str], pulp.LpVariable]]:
    teams = q1.teams
    dates = q1.dates

    m = pulp.LpProblem("Project2_Q2_FeasibleSchedule", pulp.LpMinimize)

    x: dict[tuple[str, str, str], pulp.LpVariable] = {}
    for i in teams:
        for j in teams:
            for d in dates:
                x[(i, j, d)] = pulp.LpVariable(f"x_{i}__{j}__{d}", lowBound=0, upBound=1, cat=pulp.LpBinary)

    # feasibility objective
    m += 0

    # no self-games
    for i in teams:
        for d in dates:
            m += x[(i, i, d)] == 0

    # (e) i plays home exactly on the dates computed in (a)
    for i in teams:
        for d in dates:
            m += pulp.lpSum(x[(i, j, d)] for j in teams if j != i) == q1.home_on[(i, d)]

    # (f) i plays away exactly on the dates computed in (d)
    for i in teams:
        for d in dates:
            m += pulp.lpSum(x[(j, i, d)] for j in teams if j != i) == q1.away_on[(i, d)]

    # (g) home vs counts
    for i in teams:
        for j in teams:
            if i == j:
                continue
            m += pulp.lpSum(x[(i, j, d)] for d in dates) == q1.home_vs[(i, j)]

    # (h) away at counts
    for i in teams:
        for j in teams:
            if i == j:
                continue
            m += pulp.lpSum(x[(j, i, d)] for d in dates) == q1.away_vs[(i, j)]

    return m, x


def extract_schedule_from_x(q1: Q1Data, x: dict[tuple[str, str, str], pulp.LpVariable]) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    for d in q1.dates:
        for i in q1.teams:
            for j in q1.teams:
                if i == j:
                    continue
                v = pulp.value(x[(i, j, d)])
                if v is not None and v > 0.5:
                    rows.append({"Date": d, "Home": i, "Visitor": j})

    out = pd.DataFrame(rows)
    out["Date_dt"] = pd.to_datetime(out["Date"], format="%a, %b %d, %Y")
    out = (
        out.sort_values(["Date_dt", "Home", "Visitor"])
        .drop(columns=["Date_dt"])
        .reset_index(drop=True)
    )
    return out


q2_model, x = build_q2_pulp_model(q1)
status = q2_model.solve(pulp.PULP_CBC_CMD(msg=True))

pulp.LpStatus[status]

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/sr/lsxg8d252pzc_3vgvymrxr140000gn/T/aa67ad809c804232bc5d8c24ead55ebd-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/sr/lsxg8d252pzc_3vgvymrxr140000gn/T/aa67ad809c804232bc5d8c24ead55ebd-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 1253 COLUMNS
At line 25063 RHS
At line 26312 BOUNDS
At line 30410 ENDATA
Problem MODEL has 1248 rows, 4097 columns and 15616 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0 - 0.01 seconds
Cgl0002I 3519 variables fixed
Cgl0004I processed model has 320 rows, 454 columns (454 integer (454 of which binary)) and 1386 elements
Cbc0045I No integer variables out of 454 objects (454 integer) have costs
Cbc0045I branch on satisfied N create fak

'Optimal'

In [21]:
if pulp.LpStatus[status] == "Optimal":
    schedule_df = extract_schedule_from_x(q1, x)
    schedule_path = OUT_DIR / "q2_feasible_schedule.csv"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    schedule_df.to_csv(schedule_path, index=False)
    schedule_path, schedule_df.head(10), len(schedule_df)
else:
    raise RuntimeError(f"Q2 model did not solve to Optimal. Status={pulp.LpStatus[status]}")

In [22]:
# Basic consistency checks (recommended)

# 1) Recompute Q1-style count matrices from the solved schedule and compare
re_home = pd.crosstab(schedule_df["Home"], schedule_df["Visitor"]).reindex(index=q1.teams, columns=q1.teams, fill_value=0)
re_away = pd.crosstab(schedule_df["Visitor"], schedule_df["Home"]).reindex(index=q1.teams, columns=q1.teams, fill_value=0)

orig_home = pd.DataFrame(0, index=q1.teams, columns=q1.teams, dtype=int)
orig_away = pd.DataFrame(0, index=q1.teams, columns=q1.teams, dtype=int)
for i in q1.teams:
    for j in q1.teams:
        if i == j:
            continue
        orig_home.loc[i, j] = q1.home_vs[(i, j)]
        orig_away.loc[i, j] = q1.away_vs[(i, j)]

home_ok = (re_home.values == orig_home.values).all()
away_ok = (re_away.values == orig_away.values).all()

home_ok, away_ok

(np.True_, np.True_)